# CLIPAG cohort: the 150 prod150 words, guided by CLIPAG

This notebook draws the same 150 nouns as `prod150_v1` with **CLIPAG** guidance instead of standard CLIP. Every other setting matches prod150: prompt `A drawing of {word}.`, seed 0, 256 paths, 1,000 iterations, 4 augmentations, max stroke width 50, no negative prompts, no input normalization. With seed 0 fixed, every word starts from the same initial strokes in both runs, so each CLIPAG drawing pairs with its prod150 CLIP drawing.

Results go to Google Drive in the prod150 layout (`runs/<RUN_ID>/images/part<P>/<NNN>_<word>/`), so the existing judge notebook can score them by listing that folder in `EXISTING_ROOTS`. See `clipag_cohort/README.md` for the design, the pre-registered outcomes and the analysis.

**Order of runs**
1. **Smoke test:** `RUN_MODE="smoke_test"`, `WORD_LIMIT=2`, a throwaway `RUN_ID`. It checks setup, the checkpoint and Drive output in a few minutes.
2. **CLIPAG cohort:** `RUN_MODE="full"`, `GUIDANCE_MODEL="clipag"`, `SUBSET="all"`, `RUN_ID="clipag150_v1"`. Split across teammates with `PERSON` / `NUM_PEOPLE`.
3. **CLIP control:** `GUIDANCE_MODEL="clip"`, `SUBSET="control15"`, `RUN_ID="clip150ctl_v1"`. It re-draws 15 words with standard CLIP through this code path, to check that it reproduces prod150.

Use a GPU runtime (**Runtime → Change runtime type → GPU**). If Colab disconnects, reconnect, keep the same settings and run all cells again: finished words are skipped.

In [ ]:
# @title 1. Settings
RUN_ID = "clipag150_v1"  # @param {type:"string"}
GUIDANCE_MODEL = "clipag"  # @param ["clipag", "clip"]
RUN_MODE = "full"  # @param ["full", "smoke_test"]
SUBSET = "all"  # @param ["all", "control15"]
WORD_LIMIT = 0  # @param {type:"integer"}
PERSON = 1  # @param {type:"integer"}
NUM_PEOPLE = 1  # @param {type:"integer"}
PROJECT_FOLDER = "MyDrive/ExperimentResults"  # @param {type:"string"}
SOURCE_REF = "experiment/clipdraw-clipag"  # @param {type:"string"}
REPO_URL = "https://github.com/ArmitaAfr/Visibility_Thresholds.git"

# WORD_LIMIT keeps only the first N assigned words. It is for smoke tests only (0 = all).
if WORD_LIMIT and RUN_MODE != "smoke_test":
    raise ValueError("WORD_LIMIT is only for smoke tests. Set it to 0 for a full run.")

## 2. Drive, source and word list

Mounts Drive, checks out this branch, and loads the 150 words from the exact commit and bytes prod150 used. The CLIPAG checkpoint (1.31 GB) is kept on Drive under `checkpoints/`, so later sessions skip the download.

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = (Path("/content/drive") / PROJECT_FOLDER).resolve()
if not DRIVE_ROOT.is_dir():
    raise ValueError(f"Project folder not found on Drive: {DRIVE_ROOT}")

SRC = Path("/content/Visibility_Thresholds")
if not SRC.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(SRC)], check=True)
subprocess.run(["git", "fetch", "origin", SOURCE_REF], cwd=SRC, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=SRC, check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SRC, text=True).strip()
print("Source commit:", SOURCE_COMMIT)

sys.path.insert(0, str(SRC))
import importlib
import clipag_cohort.cohort as cohort
importlib.reload(cohort)

subprocess.run(["git", "fetch", "origin", cohort.SAMPLED_WORDS_COMMIT], cwd=SRC, check=True)
words_bytes = subprocess.check_output(
    ["git", "show", f"{cohort.SAMPLED_WORDS_COMMIT}:{cohort.SAMPLED_WORDS_PATH}"], cwd=SRC)
ALL_WORDS = cohort.load_words(words_bytes)

settings = cohort.run_settings(run_id=RUN_ID, guidance_model=GUIDANCE_MODEL, run_mode=RUN_MODE,
                               subset=SUBSET, people=NUM_PEOPLE)
settings["word_limit"] = WORD_LIMIT
RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID
if RUN_ID == cohort.PROD150["run_id"] or (RUN_DIR / "stages").exists() and not (RUN_DIR / "cohort_settings.json").exists():
    raise ValueError(f"{RUN_DIR} belongs to another pipeline. Choose a new RUN_ID.")
cohort.check_or_write_json(RUN_DIR / "cohort_settings.json", settings)

words = cohort.assignment(cohort.select_words(ALL_WORDS, SUBSET), PERSON, NUM_PEOPLE)
if WORD_LIMIT:
    words = words[:WORD_LIMIT]
PART_DIR = RUN_DIR / "images" / f"part{PERSON}"
STAGING_DIR = RUN_DIR / "staging" / f"part{PERSON}"
PLAN = cohort.plan_part(words, PART_DIR, GUIDANCE_MODEL)
todo = [p for p in PLAN if not p[3]]
print(f"Run {RUN_ID} | part {PERSON}/{NUM_PEOPLE} | {len(PLAN)} words assigned, {len(PLAN) - len(todo)} done, {len(todo)} to draw")
print("First words:", [w["word"] for _, w, _, _ in PLAN[:5]])

## 3. Load the CLIPDraw / CLIPAG code

Runs the setup, configuration and helper cells of `clipdraw_clipag.ipynb` from the checked-out source, then fixes every setting to the prod150 values. The first run in a fresh runtime compiles DiffVG, which takes several minutes.

In [ ]:
import json

source_nb = json.loads((SRC / "clipdraw_clipag.ipynb").read_text(encoding="utf-8"))
code_cells = ["".join(c["source"]) for c in source_nb["cells"] if c["cell_type"] == "code"]

def cell_with(marker):
    found = [c for c in code_cells if marker in c]
    if len(found) != 1:
        raise RuntimeError(f"Expected one clipdraw_clipag.ipynb cell containing {marker!r}, found {len(found)}.")
    return found[0]

def run_cell(marker):
    exec(compile(cell_with(marker), f"clipdraw_clipag.ipynb[{marker}]", "exec"), globals())

run_cell("DIFFVG_COMMIT = ")          # environment setup and DiffVG build
run_cell("environment = {")           # runtime verification
# The configuration cell assigns its own form defaults (GUIDANCE_MODEL="clip",
# RUN_MODE="full"), so keep this notebook's choices and restore them afterwards.
chosen = {"GUIDANCE_MODEL": GUIDANCE_MODEL, "RUN_MODE": RUN_MODE}
run_cell('EXPERIMENT_MODE = "single"')  # defaults; overridden below
GUIDANCE_MODEL, RUN_MODE = chosen["GUIDANCE_MODEL"], chosen["RUN_MODE"]
if (GUIDANCE_MODEL, RUN_MODE) != (settings["guidance_model"], settings["run_mode"]):
    raise RuntimeError("Settings changed while loading the source notebook.")

# Fix every setting to prod150. "cohort" mode keeps the later cells from auto-running.
EXPERIMENT_MODE = "cohort"
P = cohort.PROD150
PROMPT = P["prompt_template"].format(words[0]["word"]) if words else "A drawing of nothing."
SEED = P["seed"]
NUM_PATHS, NUM_ITERATIONS, NUM_AUGMENTATIONS = P["num_paths"], P["num_iterations"], P["num_augmentations"]
MAX_STROKE_WIDTH = P["max_stroke_width"]
USE_NEGATIVE_PROMPTS = P["use_negative_prompts"]
NORMALIZE_CLIP_INPUTS = P["normalize_clip_inputs"]
ENABLE_PREDICTION = False
DISPLAY_EVERY = 50
CLIPAG_CHECKPOINT_PATH = str(DRIVE_ROOT / "checkpoints" / "CLIPAG_ViTB32.pt")
DOWNLOAD_CLIPAG_CHECKPOINT = True
effective = {
    "num_paths": 16 if RUN_MODE == "smoke_test" else NUM_PATHS,
    "num_iterations": 5 if RUN_MODE == "smoke_test" else NUM_ITERATIONS,
    "num_augmentations": 1 if RUN_MODE == "smoke_test" else NUM_AUGMENTATIONS,
}

run_cell("def load_guidance_model")   # guidance loading and rendering helpers
run_cell("def initialize_scene")      # scene initializer
run_cell("def optimize_scene")        # optimizer
run_cell("def export_run")            # exporter
print("Loaded clipdraw_clipag.ipynb code. Effective workload:", effective)

## 4. Draw the assigned words

Loads the guidance model once, then draws each remaining word. Each result is written to a staging folder and moved into `images/` only once complete, so an interrupted run never leaves a partial result. Every word's initial strokes are hashed and checked against the run's first word, which confirms the pairing with prod150.

In [ ]:
import csv, gc, hashlib, shutil, time
import numpy as np
import torch

# Clear staging left by an interrupted session; those words were never published.
if STAGING_DIR.exists():
    shutil.rmtree(STAGING_DIR)
STAGING_DIR.mkdir(parents=True)
LOG = RUN_DIR / "logs" / f"part{PERSON}.csv"
LOG.parent.mkdir(parents=True, exist_ok=True)
log_fields = ["utc", "folder", "word", "status", "optimization_seconds", "final_loss", "error"]
if not LOG.exists():
    with LOG.open("w", newline="") as f:
        csv.DictWriter(f, fieldnames=log_fields).writeheader()

def log(row):
    with LOG.open("a", newline="") as f:
        csv.DictWriter(f, fieldnames=log_fields).writerow(
            {"utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), **row})

failures = []
if todo:
    load_guidance_model()
    print(json.dumps(guidance_metadata, indent=2))
for n, (index, w, folder, _) in enumerate(todo, 1):
    word = w["word"]
    batch_progress = f"{RUN_ID} | part {PERSON} | {n}/{len(todo)} | {word} ({w['group']})"
    print(batch_progress)
    try:
        PROMPT = P["prompt_template"].format(word)
        prepare_guidance()
        initialize_scene()
        initial_hash = hashlib.sha256(initial_svg.encode()).hexdigest()
        # Seed 0 and the same path count give identical initial strokes for every word.
        cohort.check_or_write_json(RUN_DIR / "initial_scene.json",
                                   {"sha256": initial_hash, "num_paths": effective["num_paths"], "seed": SEED})
        optimize_scene()
        if not all(np.isfinite(v) for v in loss_history):
            raise RuntimeError("Nonfinite training loss.")
        result = export_run(STAGING_DIR, show=False)
        record = cohort.cohort_record(result["experiment"], word_row=w, index=index, settings=settings,
                                      person=PERSON, source_commit=SOURCE_COMMIT)
        record["experiment_mode"] = "cohort"
        Path(result["metadata_path"]).write_text(json.dumps(record, indent=2))
        cohort.publish(result["output_dir"], folder)
        log({"folder": folder.name, "word": word, "status": "ok", "optimization_seconds": round(elapsed_seconds, 1),
             "final_loss": round(loss_history[-1], 6), "error": ""})
    except Exception as exc:
        failures.append((word, f"{type(exc).__name__}: {exc}"))
        log({"folder": folder.name, "word": word, "status": "error", "optimization_seconds": "",
             "final_loss": "", "error": failures[-1][1]})
        print("FAILED:", failures[-1])
    finally:
        gc.collect()
        torch.cuda.empty_cache()
batch_progress = None

PLAN = cohort.plan_part(words, PART_DIR, GUIDANCE_MODEL)
remaining = [w["word"] for _, w, _, done in PLAN if not done]
if remaining:
    print(f"{len(remaining)} words not finished: {remaining}. Rerun this notebook to retry them.")
else:
    receipt = {"run_id": RUN_ID, "person": PERSON, "folders": [f.name for _, _, f, _ in PLAN],
               "png_sha256": {f.name: hashlib.sha256((f / "clipdraw_final.png").read_bytes()).hexdigest()
                              for _, _, f, _ in PLAN}}
    cohort.check_or_write_json(RUN_DIR / "generation" / f"part{PERSON}.json", receipt)
    print(f"Part {PERSON} complete: {len(PLAN)} drawings in {PART_DIR}")

## 5. Next: judge the drawings

When every part has a receipt in `runs/<RUN_ID>/generation/`, score the images with the team's judge notebook (`visibility_thresholds_colab.ipynb`): a new evaluation `RUN_ID`, `EXISTING_ROOTS = [DRIVE_ROOT / "runs/<RUN_ID>/images"]`, `SOURCE_LINEAGE = None`, then `MANIFEST` → `GPU_EVALUATE` → `CPU_EVALUATE`. Compare its `analysis/combined.csv` with prod150's using `clipag_cohort/compare.py` (see the README).